# 01 - The data: which corpora exist, who wrote them, and whether they are clean

**What this notebook is for.** This is the front door of the report series. Every later
notebook (geometry, detection probes, transition tracking) computes its numbers from text
corpora and from the activations recorded while a model reads them. Before any of those
results can be judged, three questions have to be settled here: what text exists, who wrote
it, and is it clean enough to build on. Nothing in this notebook is a research finding; it is
the inventory and the quality control that every finding rests on.

**How to read this project.** It has three layers, and keeping them apart prevents most
misreadings:

<table>
<tr><td><b>the notebooks</b></td><td>the <b>report</b>: readable, in story order, every figure
carrying its own verdict and its own grading scale</td></tr>
<tr><td><b><code>results/</code></b></td><td>the <b>evidence</b>: every number printed in a
notebook traces back to a file here (small JSON in git, bulky arrays on Hugging Face)</td></tr>
<tr><td><b><a href="../TREE.md">TREE.md</a></b></td><td>the <b>record</b>: questions,
hypotheses, experiments and claims, with predictions registered before the data existed. When
a notebook and the tree disagree, the tree wins and the notebook has a bug.</td></tr>
</table>

[`notebooks/README.md`](README.md) holds the reading order for the whole series and the setup
steps (which kernel, where the `HF_TOKEN` goes); this notebook does not repeat them.

**Where the data comes from.** No path here is machine specific: every input resolves through
`emotion_vectors.artifacts.fetch()`, which looks in the local `results/` tree first and pulls
from the project's Hugging Face datasets otherwise, so this notebook runs unchanged on any
clone. [`DATA.md`](../DATA.md) is the human-readable index of those datasets; its
machine-readable twin is the `ROUTES` table in `src/emotion_vectors/artifacts.py`. All figure
and analysis code lives in the importable package `emotion_vectors.corpora_report`; the cells
below only import, call, and narrate.

**Key concepts** (plain words first, the registry name in parentheses):

- *Corpus*: one file of short texts, each written to evoke one target emotion. Stored one row
  per emotion, with that emotion's texts inside the row (registry name: a *grouped jsonl*).
- *Generator* versus *reader*: the **generator** is the model that WROTE a corpus. The
  **reader** is the model whose internal activations were recorded while it READ that corpus.
  They are usually different models here, and confusing them is the fastest way to misread
  every later figure. Section 1 names the generator of every corpus; section 3 names the
  reader of every vector bundle.
- *Emotion vector*: the average of a reader model's internal activations (its residual stream)
  over all the texts for one emotion, at one layer. This is the object every later notebook
  actually studies.
- *Vector bundle*: the output of one extraction run: per-text activation shards, the
  per-emotion mean vectors, and a `manifest.jsonl` recording one row per text (its emotion,
  its token count, and an error field that is empty when extraction succeeded).
- *Leakage*: a generated text naming its own target emotion in words, despite the generation
  instruction not to. Low leakage matters because a probe built on leaky text can succeed by
  detecting a word rather than the feeling behind it (registry name: the stem lists in
  `emotion_vectors.scoring.EMOTION_STEMS`).
- *Battery*: the short list of emotions used for the probe experiments, the ones that have
  stem lists and therefore can be leakage-audited. Section 2 prints how many there are.
- *Blessed instrument*: the vector bundles whose names end in `-postfix`, re-extracted after a
  padding bug was found and fixed. The pre-fix bundles are kept unchanged as the historical
  record for the before-and-after audit (TREE node Q1.H3.E4b).
- *HF*: [Hugging Face](https://huggingface.co/abotresol), where the datasets are published.
  Some are already public and some, including the blessed bundles, stay private to the team
  until the end of the sprint; [`DATA.md`](../DATA.md) marks which is which.

**Index.**

1. [What text does every downstream result rest on?](#1) The corpus catalog: one row per
   corpus, with its generator, its role, and its measured size.
2. [Do the generated texts avoid naming their target emotion?](#2) The leakage audit, and the
   one corpus that fails it.
3. [Are the blessed vector bundles complete?](#3) Extraction integrity, which reader produced
   each bundle, and where each bundle is published.

<a id="1"></a>

## 1. What text does every downstream result rest on?

Plain question: which files of text exist in this project, who wrote each one, how big it is,
and which experiments consume it. The cell below loads every corpus once, and every later
section reads from that one load, so the catalog, the leakage audit, and the printed record
cannot disagree about the contents of a file.

The corpora come from four different generators. One is external and not ours: the story corpus
of the open replication of the source paper ([Anthropic's emotions
paper](https://transformer-circuits.pub/2026/emotions),
[arXiv 2604.07729](https://arxiv.org/abs/2604.07729)), written by gemma-4-4B and published as
[`snae/emotion_stories_gemma_4_4B`](https://huggingface.co/datasets/snae/emotion_stories_gemma_4_4B).
That corpus is the reference point the leakage audit in section 2 grades everything else
against, because it is the text the published work itself was built on.
The rest we generated: with the instruction-tuned model
[`google/gemma-4-31b-it`](https://huggingface.co/google/gemma-4-31b-it), with its untuned
counterpart [`google/gemma-4-31b`](https://huggingface.co/google/gemma-4-31b), and with an
external commercial model, `deepseek-v4-pro`, called through
[OpenRouter](https://openrouter.ai/). Their published homes, in catalog order, are
[emotion-stories-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/emotion-stories-gemma-4-31b-it),
[emotion-dialogues-gemma-4-31b](https://huggingface.co/datasets/abotresol/emotion-dialogues-gemma-4-31b)
and [-it](https://huggingface.co/datasets/abotresol/emotion-dialogues-gemma-4-31b-it),
[neutral-transcripts-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/neutral-transcripts-gemma-4-31b-it),
[emotion-combined-stories-gemma-4-31b-it](https://huggingface.co/datasets/abotresol/emotion-combined-stories-gemma-4-31b-it),
[emotion-stories-deepseek-v4-pro](https://huggingface.co/datasets/abotresol/emotion-stories-deepseek-v4-pro),
and [emotion-stories-deepseek-v4-pro-diverse](https://huggingface.co/datasets/abotresol/emotion-stories-deepseek-v4-pro-diverse).

In [1]:
# this cell loads every story corpus in the project exactly once, through
# emotion_vectors.artifacts.fetch (local results/ first, the published Hugging Face
# dataset otherwise); all three sections read from this one context, so no two of
# them can report different sizes for the same file
from emotion_vectors.corpora_report import REFERENCE_CORPUS_ID, load_corpora_context

ctx = load_corpora_context()
print("\n".join(ctx.lines))

# LOAD-BEARING VERIFICATION (keep this block): the two anchors that pin this catalog to
# the datasets DATA.md documents. The reference corpus must be the open replication's
# 171-emotion story corpus, and the Q3 combined corpus must hold the 173-triple recipe.
# If either assert fires, a file on disk is not the corpus this report describes, and
# every count below it is describing something else.
reference = ctx.entry("published")
assert reference.location == REFERENCE_CORPUS_ID and reference.n_groups == 171, (
    f"reference corpus is {reference.location} covering {reference.n_groups} emotions"
)
assert ctx.combined_triples == 173, f"combined corpus holds {ctx.combined_triples} triples"
print(
    f"\nanchors OK: reference corpus {reference.location} covers {reference.n_groups} "
    f"emotions; the combined (Q3) corpus holds {ctx.combined_triples} emotion triples"
)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


published stories (reference): 171 emotions, 1,539 texts (snae/emotion_stories_gemma_4_4B)
self stories, instruct: 12 emotions, 3,072 texts (results/self_stories_it/dialogues_grouped.jsonl)
dialogues, base model: 12 emotions, 192 texts (results/dialogue_stories/dialogues_grouped.jsonl)
dialogues, instruct: 12 emotions, 192 texts (results/dialogue_stories_it/dialogues_grouped.jsonl)
neutral transcripts: 1 emotions, 128 texts (results/neutral_transcripts_it/dialogues_grouped.jsonl)
combined stories (Q3): 171 emotions, 5,888 texts (results/combined_stories/stories_grouped.jsonl)
DeepSeek, fixed prompt: 12 emotions, 3,070 texts (results/openrouter_stories/stories_grouped.jsonl)
DeepSeek, diverse prompts: 12 emotions, 12,262 texts (results/openrouter_stories_diverse/stories_grouped.jsonl)

anchors OK: reference corpus snae/emotion_stories_gemma_4_4B covers 171 emotions; the combined (Q3) corpus holds 173 emotion triples


In [2]:
# section 1 is load-call-show over emotion_vectors.corpora_report: the census table, with
# every size column counted from the file named in that same row's location column
from emotion_vectors.corpora_report import s1_catalog_figure

fig_s1, s1_stats = s1_catalog_figure(ctx)
fig_s1.show()
print("\n".join(s1_stats["lines"]))

catalog totals: 8 corpora, 26,343 texts, 4 generator models (deepseek-v4-pro, gemma-4-31b base, gemma-4-31b-it, gemma-4-4B)
combined (Q3) corpus shape caveat: its file groups stories by emotion triple (173 triples), so its emotions column counts distinct emotions across triples, not rows of the file


<details><summary><b>How to read this table</b></summary>

One row is one corpus. Read a row left to right: its plain name, the file it lives in (a
`results/`-relative path, or a Hugging Face dataset id when there is no local copy), the model
that WROTE the texts, the experiments that consume it, and two counts. The *emotions* column is
how many distinct target emotions the file covers. The *texts* column is how many individual
stories or dialogues it holds. Both counts are computed from the file by the cell above and
printed underneath the table, so they can be checked against the file itself rather than taken
on trust.

**Valid readings.** Comparing the texts column across rows tells you which corpora are large
enough for a scale argument and which are pilots. Reading the generator column tells you whose
writing style is baked into a corpus, which is exactly what the generator-lineage experiments
(registry names: E11 and E12) manipulate on purpose.

**Invalid readings.** Do not read the generator column as "the model being studied": the
generator wrote the text, and the reader model whose activations were recorded is a separate,
per-experiment choice, named for each bundle in section 3. Do not compare the combined stories
(Q3) row's emotions count with the other rows as if it meant the same thing: that file groups
its texts by emotion *triple* rather than by single emotion, so its emotions column counts the
distinct emotions appearing across the triples, which the printed caveat line under the table
states along with the triple count.

**Grading.** This exhibit has no pass or fail line, and that is deliberate: a census cannot be
right or wrong about a threshold, only complete or incomplete. A good result here is simply
that every corpus a later notebook cites appears in this table with a size counted from its own
file; a bad result would be a corpus used downstream that has no row here. Cleanliness is graded
in section 2, and extraction integrity in section 3.

</details>

**What this section establishes.** The complete inventory of text this project rests on, with
each corpus's size measured rather than asserted, and each corpus's generator named so that no
later figure can quietly confuse who wrote a text with who read it. It establishes nothing about
quality: a corpus can be large and still be unusable.

**Open question, taken up next.** These texts were all generated under an instruction not to
name the target emotion. Whether the generators actually obeyed is an empirical question, and it
decides whether a probe trained on a corpus could be detecting a word instead of a feeling.
Section 2 measures it.

<a id="2"></a>

## 2. Do the generated texts avoid naming their target emotion?

Every corpus above was generated with an instruction of the form "write a story where the
character feels this way, without naming the emotion". If a text names its emotion anyway, then
a probe fitted on that corpus can succeed by picking up the word, and its success would say
nothing about whether the model represents the feeling. That failure mode is called *leakage*,
and this section measures it corpus by corpus.

The measurement uses the same word-stem lists the scoring pipeline uses
(`emotion_vectors.scoring.EMOTION_STEMS`), so this audit and the probe experiments agree on what
"naming the emotion" means. Only the battery emotions have stem lists, so a corpus covering more
emotions than the battery is audited on the battery subset of its texts; the cell below prints
that arithmetic explicitly, because the audit denominator is smaller than the corpus size in
section 1 and the difference is easy to mistake for a contradiction.

In [3]:
# section 2 is load-call-show: leakage per corpus, colored by who WROTE the texts, with
# the reference-corpus level and the 0% floor drawn as the grading band
from emotion_vectors.corpora_report import s2_leakage_figure

fig_s2, s2_stats = s2_leakage_figure(ctx)
fig_s2.show()
print("\n".join(s2_stats["lines"]))

# DENOMINATOR CROSS-CHECK (keep this block): section 1 counted the whole reference corpus,
# section 2 audits only its battery subset. Print the arithmetic that connects the two
# numbers, so a reader comparing them does not read a contradiction into the report.
reference_leaked, reference_audited = s2_stats["audited"][reference.label]
texts_per_emotion = reference.n_texts // reference.n_groups
assert s2_stats["n_battery_emotions"] * texts_per_emotion == reference_audited, (
    "the audited denominator is not the battery subset of the reference corpus"
)
print(
    f"\ndenominator cross-check: the reference corpus holds {reference.n_texts:,} texts over"
    f" {reference.n_groups} emotions ({texts_per_emotion} per emotion), but only the"
    f" {s2_stats['n_battery_emotions']} battery emotions have stem lists, so the audit"
    f" denominator is {s2_stats['n_battery_emotions']} x {texts_per_emotion} ="
    f" {reference_audited}, not {reference.n_texts:,}"
)

published stories (reference): 4/108 audited texts name their emotion (3.7%)
dialogues, base model: 84/192 audited texts name their emotion (43.8%)
dialogues, instruct: 0/192 audited texts name their emotion (0.0%)
self stories, instruct: 42/3072 audited texts name their emotion (1.4%)
DeepSeek, fixed prompt: 26/3070 audited texts name their emotion (0.8%)
DeepSeek, diverse prompts: 123/12262 audited texts name their emotion (1.0%)
audited = texts whose target emotion is one of the 12 battery emotions with stem lists, so a corpus covering more emotions (the reference corpus covers 171) is audited on that battery subset only

denominator cross-check: the reference corpus holds 1,539 texts over 171 emotions (9 per emotion), but only the 12 battery emotions have stem lists, so the audit denominator is 12 x 9 = 108, not 1,539


In [4]:
# LOAD-BEARING VERIFICATION (keep this block): this section's claim is an ordering, not a
# single number. Every instruction-following corpus must sit at or below the reference
# corpus the source papers worked with, and the base-model corpus must sit far above it.
# If this ordering ever flips, the "base arm carries a lexical confound" caveat that the
# probe notebooks inherit from here is no longer supported by anything.
base_percent = s2_stats["base_percent"]
reference_percent = s2_stats["reference_percent"]
instruction_following = {
    label: pct for label, pct in s2_stats["percent"].items() if label != s2_stats["base_label"]
}
assert max(instruction_following.values()) <= reference_percent, (
    f"an instruction-following corpus leaks above the reference level: {instruction_following}"
)
assert base_percent > 10 * reference_percent, (
    f"base corpus at {base_percent:.1f}% is no longer far above {reference_percent:.1f}%"
)
print(
    f"ordering OK: every instruction-following corpus at or below {reference_percent:.1f}%"
    f" (the reference corpus level), the base-model corpus at {base_percent:.1f}%,"
    f" a factor of {base_percent / reference_percent:.0f} higher"
)

ordering OK: every instruction-following corpus at or below 3.7% (the reference corpus level), the base-model corpus at 43.8%, a factor of 12 higher


<details><summary><b>How to read this figure</b></summary>

Each bar is one corpus, in the same naming as the catalog table above. The horizontal axis names
the corpus; the vertical axis is the percentage of that corpus's *audited* texts (the battery
subset, per the cross-check printed above) that contain a word-stem of their own target emotion.
Lower is better, because the instruction was to not name it. Bar color encodes who WROTE the
texts, and the legend spells that role out: these are generator identities, not reader models,
and no reader model appears in this figure at all.

**The grading scale is drawn in the plot.** The dotted line at the bottom is the floor of the
scale: 0% means every text obeyed. The dashed green line is the level of the reference corpus,
the corpus the source papers themselves worked with, and the shaded green band beneath it is
therefore the "clean enough to reuse" region. Note the direction: the reference line is the
*ceiling* of the acceptable set, not a floor everyone beats. The red annotation marks the
failure anchor.

**Valid readings.** A bar inside the green band means that corpus is at least as clean as the
published corpus the field already builds on, so a probe fitted on it is unlikely to be reading
words. A bar far above the band means the opposite, and the base-model bar is the one such bar:
it is the reason every base-arm dialogue result carries a lexical-confound caveat in the probe
notebooks.

**Invalid readings.** Do not read a low bar as evidence that a corpus is *good*, only that it is
not leaky: leakage is one failure mode out of many, and says nothing about whether the texts are
varied, vivid, or on-target. Do not compare bar heights to the raw corpus sizes in section 1: the
denominators here are the battery subsets, printed above. Do not read the base-model bar as a
defect in the base model as a reader; it is a fact about the base model as a *writer*, which
follows instructions poorly.

**A good result, a bad result, and where this sits.** A good result would be every bar inside the
green band. A bad result would be several bars at the base model's height, which would put a
lexical confound under most of the project. What is observed is neither: every
instruction-following corpus sits inside the band (the printed ordering check states by how
much), and exactly one corpus, the base-model dialogues, sits far above it and is quarantined
with a caveat rather than deleted.

</details>

**What this section establishes.** The corpora generated by instruction-tuned models are at
least as clean, by this measure, as the published corpus the source papers used, so downstream
probe results built on them are not trivially explainable as word detection. It also establishes
the single known exception: the base-model dialogue corpus, whose leakage is high enough that any
result derived from it must carry the lexical-confound caveat.

**Live hypothesis (labeled as a hypothesis, not a finding).** The base model's leakage is a
failure of instruction following rather than a property of dialogue as a format. The comparison
that decides it is already in the catalog: both dialogue corpora were generated by the same
script with the same instruction text (`scripts/generate_dialogue_stories.py`, style `dialogue`,
whose instruction says in as many words not to name the emotion anywhere in the text), wrapped in
each generator's own input convention, so format is held fixed and only the generator changes.
Their two bars are the test, and the printed record above shows where each one sits.

**Open questions.** Word-stem matching is a blunt instrument: it catches "afraid" but not "her
hands would not stop shaking". A paraphrase-level leakage audit would be a stronger test and has
not been run. Nothing in this section tells us whether the clean corpora are *informative*, only
that they are not leaky.

<a id="3"></a>

## 3. Are the blessed vector bundles complete?

The corpora above are only the input. What the later notebooks actually load are *vector
bundles*: the result of running a reader model over a corpus (`src/emotion_vectors/extraction.py`,
driven by `scripts/extract_emotion_vectors.py`) and saving, per text, the activations at a fixed
set of layers, plus the per-emotion means. This section asks the one question that decides whether
a bundle can be used at all: did every text extract successfully, or are there silent gaps.

**Which bundles are blessed.** A padding bug was found and fixed partway through the sprint, so
every bundle was re-extracted; the re-extractions carry the suffix `-postfix` and are the only
bundles any current result should use. The pre-fix bundles are deliberately kept unchanged as the
historical record, because the before-and-after comparison is itself an experiment (TREE node
Q1.H3.E4b), and each published `-postfix` dataset carries a `LINEAGE.md` recording the fix.

**Where each bundle lives.** All are private team datasets on Hugging Face:

<table>
<tr><th align="left">bundle</th><th align="left">what it holds</th></tr>
<tr><td><a href="https://huggingface.co/datasets/abotresol/emotion-vectors-gemma-4-31b-postfix">emotion-vectors-gemma-4-31b-postfix</a></td>
<td>base reader over the reference corpus</td></tr>
<tr><td><a href="https://huggingface.co/datasets/abotresol/emotion-vectors-gemma-4-31b-it-postfix">emotion-vectors-gemma-4-31b-it-postfix</a></td>
<td>instruct reader over the reference corpus</td></tr>
<tr><td><a href="https://huggingface.co/datasets/abotresol/emotion-selfstory-vectors-gemma-4-31b-it-postfix">emotion-selfstory-vectors-gemma-4-31b-it-postfix</a></td>
<td>instruct reader over its own self-generated stories</td></tr>
<tr><td><a href="https://huggingface.co/datasets/abotresol/neutral-vectors-gemma-4-31b-it-postfix">neutral-vectors-gemma-4-31b-it-postfix</a></td>
<td>instruct reader over the neutral transcripts (the no-emotion control)</td></tr>
</table>

Every other evidence file the report notebooks cite lives in the catch-all
[experiment-artifacts dataset](https://huggingface.co/datasets/abotresol/emotion-vectors-experiment-artifacts);
[`DATA.md`](../DATA.md) indexes the pre-fix sets, the dialogue-lineage and DeepSeek-lineage
vector sets, and the per-token trajectory substrates that the trajectory explorer notebooks
(05, 06 and 09) read.

In [5]:
# section 3 is load-call-show: each blessed bundle sized from its own manifest.jsonl and
# described by its own run_config.json (which reader model, which source corpus, how many
# layers), so the reader-versus-generator distinction is visible in the table itself
from emotion_vectors.corpora_report import s3_bundles_figure

fig_s3, s3_stats = s3_bundles_figure()
fig_s3.show()
print("\n".join(s3_stats["lines"]))

results/emotion_vectors_postfix: 171 emotions, 1,539 stories, 199,523 tokens, 20 layers, 0 errors (reader google/gemma-4-31b, source snae/emotion_stories_gemma_4_4B)
results/emotion_vectors_it_postfix: 171 emotions, 1,539 stories, 197,984 tokens, 20 layers, 0 errors (reader google/gemma-4-31b-it, source snae/emotion_stories_gemma_4_4B)
results/self_story_vectors_it_postfix: 12 emotions, 3,072 stories, 375,842 tokens, 20 layers, 0 errors (reader google/gemma-4-31b-it, source results/self_stories_it/dialogues_grouped.jsonl)
results/neutral_vectors_it_postfix: 1 emotions, 128 stories, 15,658 tokens, 20 layers, 0 errors (reader google/gemma-4-31b-it, source results/neutral_transcripts_it/dialogues_grouped.jsonl)
bundle totals: 6,278 stories, 0 extraction errors


In [6]:
# LOAD-BEARING VERIFICATION (keep this block): two integrity anchors for the instrument.
# First, no extraction may have failed: a nonzero errors count means some emotion's mean
# vector was averaged over fewer texts than the manifest claims, silently.
# Second, every bundle's source corpus must be a row of the section-1 catalog, which is
# what makes this notebook a closed chain from text to vector with no unexplained inputs.
assert s3_stats["total_errors"] == 0, (
    f"{s3_stats['total_errors']} extraction errors in the blessed bundles"
)
catalog_locations = {row.location for row in ctx.catalog}
bundle_sources = {source for _, _, source, *_ in s3_stats["rows"]}
unlisted_sources = bundle_sources - catalog_locations
assert not unlisted_sources, f"bundle sources missing from the catalog: {sorted(unlisted_sources)}"
print(
    f"integrity OK: {s3_stats['total_errors']} extraction errors, and all"
    f" {len(bundle_sources)} distinct bundle source corpora appear in the section-1 catalog"
)

integrity OK: 0 extraction errors, and all 3 distinct bundle source corpora appear in the section-1 catalog


<details><summary><b>How to read this table</b></summary>

One row is one extraction run. The first column is where the bundle lives under `results/`. The
next two are the pair that matters most: the **reader model**, whose activations were recorded,
and the **source corpus**, the text it read. Those two are independent, and the table shows it:
the same reference corpus was read by two different reader models, producing two different
bundles. The remaining columns are counted from the run's own files: how many emotions have mean
vectors, how many texts were extracted, how many tokens those texts came to, at how many layers
each vector was saved, and how many texts failed.

**The grading scale is the errors column.** A table cannot carry reference lines, so the pass or
fail lives in the cell color: a green cell reading 0 PASS means every text in the manifest
extracted successfully; a red cell reading FAIL would mean some emotion's mean vector was
averaged over fewer texts than intended, without anything downstream noticing. The assert
underneath the figure is the same check, so a future re-extraction with a gap halts the notebook
rather than quietly changing every number in the report.

**Valid readings.** The tokens column is a rough sense of how much text each mean vector is
averaged over, which matters when comparing bundles of very different sizes. The source corpus
column can be matched, string for string, against the location column of the section-1 catalog;
the assert above enforces exactly that, so this notebook is a closed chain from text file to
vector bundle.

**Invalid readings.** Do not read the reader model column as the generator: for the first two
rows, gemma-4-31b and gemma-4-31b-it are reading stories that neither of them wrote. Do not read
zero errors as a claim that the vectors are *good*: it is only a claim that they are complete.
Whether the vectors carry emotion structure is notebook 02's question, and whether they detect
emotion is notebook 03's.

**A good result, a bad result, and where this sits.** A good result is a zero in every errors
cell with the stories counts matching the corresponding corpus sizes in section 1. A bad result
would be any nonzero errors cell, which would disqualify that bundle as the blessed instrument
until it was re-extracted. What is observed is the good case, on every bundle, as the printed
totals state.

</details>

**What this section establishes.** The instrument every later notebook loads is complete and
traceable: each blessed bundle extracted every text it claims, each names the reader model that
produced it and the corpus it read, and each of those corpora is a row of the section-1 catalog.
Together with section 2 that closes the front-door question: the text exists, its provenance is
recorded, one corpus is known to be leaky and is flagged, and the vectors derived from all of it
have no silent gaps.

**Live hypothesis (labeled as a hypothesis).** The padding fix changed the vectors but not the
conclusions drawn from them. This notebook cannot decide that, because completeness is not
agreement: the deciding evidence is the before-and-after audit `results/e4b_extraction_impact.json`,
whose per-layer comparison is asserted against the running bundles at the top of notebook 02.

**Open questions.** Token counts vary widely across bundles, and whether mean vectors from a
short-text corpus are comparable with mean vectors from a long-text one is not settled here;
notebook 02 returns to text length as a possible confound. Nothing here checks that the texts are
*about* the emotions they are labeled with, only that they were extracted; that labeling
assumption is inherited from the generation prompts.

**Where to go next.** Notebook 02 asks whether these vectors reproduce the published geometry of
emotion space. Notebook 03 asks whether they can detect an implied emotion in a scenario. Notebook
07 asks whose stories make the best probes, which is where the generator column of the section-1
catalog stops being bookkeeping and becomes the independent variable. The full reading order is in
[`notebooks/README.md`](README.md).